# Listening Past the Noise — Training

Extracts acoustic features, fits unsupervised patch encoders, runs 3-seed 5-fold cross-validation for calibration, trains the final 9-stream ensemble on all training data, and saves model artifacts.

## Pipeline overview
1. **Artifact loading:** Locates and loads `meta_and_encoders.joblib`, `models.joblib`, and `train_refs.npz` exported by the training notebook.
2. **Test feature extraction:** Applies the same 6-view acoustic extraction function used during training to all recordings in `test.csv` / `audio/test/`, imputing any missing values with the saved training medians.
3. **Patch transformation:** Encodes test short-window patches into BoAW, VLAD, and Fisher vector representations using the pre-fitted scalers, k-means codebooks, PCA projections, and GMMs (transform only, no refitting).
4. **Kernel computation and prediction:** Computes test-against-train linear and RBF kernels using the saved training reference matrices, generates probability predictions from all 9 saved streams (including similarity-weighted verified 16-NN), and combines them with the fixed ensemble weights.
5. **Calibration and submission:** Divides out the saved OOF class marginal (`pi_src`), applies temperature scaling (`temp_opt`) and damped EM test-prior adjustment (`power = 0.50`), and writes the top-3 predicted categories per recording to `/kaggle/working/submission.csv`.

## Execution
Attach both the `project4` dataset and the saved output of the training notebook as inputs, then run all cells (`Save Version -> Save & Run All`).

In [ ]:
import os, gc, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import joblib
from scipy.signal import periodogram
from scipy.spatial.distance import cdist
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics.pairwise import linear_kernel, rbf_kernel

warnings.filterwarnings("ignore")
T_GLOBAL = time.time()

if Path("/kaggle/input").exists() and list(Path("/kaggle/input").rglob("train.csv")):
    INPUT_ROOT = Path(os.environ.get("P4_INPUT", "/kaggle/input"))
    WORK = Path(os.environ.get("P4_WORK", "/kaggle/working"))
else:
    INPUT_ROOT = Path(os.environ.get("P4_INPUT", "project4"))
    WORK = Path(os.environ.get("P4_WORK", "work_step2b_train"))
WORK.mkdir(parents=True, exist_ok=True)
ART_DIR = WORK / "step2b_artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)

N_JOBS = int(os.environ.get("P4_JOBS", -1))
SEEDS = [42, 913, 2026]

train_csvs = [p for p in INPUT_ROOT.rglob("train.csv") if "step2b_artifacts" not in str(p)]
assert len(train_csvs) == 1, f"Expected 1 train.csv under {INPUT_ROOT}, found {train_csvs}"
DATA = train_csvs[0].parent
train_df = pd.read_csv(DATA / "train.csv", dtype={"id": str})
TRAIN_AUDIO = list(DATA.rglob("audio/train"))[0] if list(DATA.rglob("audio/train")) else list(INPUT_ROOT.rglob("audio/train"))[0]
assert train_df["id"].is_unique

classes = np.array(sorted(train_df["label"].unique()))
K = len(classes)
cls_idx = {c: i for i, c in enumerate(classes)}
verified_mask = (train_df["checked"].values == 1)
y_all_idx = train_df["label"].map(cls_idx).values
yv = y_all_idx[verified_mask]
yu = y_all_idx[~verified_mask]

n_tr = len(train_df)
ver_ids = np.flatnonzero(verified_mask)
unc_ids = np.flatnonzero(~verified_mask)
ver_cnt = np.bincount(yv, minlength=K)
tot_cnt = np.bincount(y_all_idx, minlength=K)
w_unif  = (1.0 / ver_cnt)[yv]
w_tot   = ((tot_cnt / tot_cnt.sum()) / (ver_cnt / ver_cnt.sum()))[yv]
v40     = (ver_cnt == ver_cnt.max())

print(f"data: {DATA} | artifacts: {ART_DIR}")
print(f"train: {n_tr} (verified={len(yv)}, unchecked={len(yu)}) | classes: {K}")

## Feature extraction and patch encoding

In [ ]:
SR, N_FFT, HOP = 22050, 2048, 512
MEL32_2048 = librosa.filters.mel(sr=SR, n_fft=2048, n_mels=32).astype(np.float32)
MEL24_1024 = librosa.filters.mel(sr=SR, n_fft=1024, n_mels=24).astype(np.float32)
MEL40_1024 = librosa.filters.mel(sr=SR, n_fft=1024, n_mels=40).astype(np.float32)
MEL16_1024 = librosa.filters.mel(sr=SR, n_fft=1024, n_mels=16).astype(np.float32)

def count_onsets(env, sr, hop):
    if not np.isfinite(env).all() or not np.any(env):
        return 0
    normalized = env - env.min()
    normalized = normalized / (normalized.max() + np.finfo(env.dtype).tiny)
    pre_max = int(0.03 * sr // hop)
    pre_avg = int(0.10 * sr // hop)
    post_avg = pre_avg + 1
    wait = pre_max
    count = 0
    previous = -wait - 1
    for i, value in enumerate(normalized):
        if i <= previous + wait:
            continue
        peak_window = normalized[max(0, i - pre_max):i + 1]
        avg_window = normalized[max(0, i - pre_avg):i + post_avg]
        if value == peak_window.max() and value >= avg_window.mean() + 0.07:
            count += 1
            previous = i
    return count

def mask_slice(M, mask):
    n = min(M.shape[1], len(mask))
    m = mask[:n]
    if m.sum() < 2:
        m = np.ones(n, bool)
    return M[:, :n][:, m]

def make_window_patches(mat, widths=(8, 32)):
    views = []
    for w in widths:
        h = w // 2
        fr = np.pad(mat, ((0, max(0, w - len(mat))), (0, 0)), mode="edge")
        starts = list(range(0, len(fr) - w + 1, h))
        if starts[-1] != len(fr) - w:
            starts.append(len(fr) - w)
        bl = np.stack([fr[i:i + w] for i in starts])
        hf = w // 2
        views.append(np.hstack([bl.mean(1), bl.std(1), bl[:, hf:].mean(1) - bl[:, :hf].mean(1)]).astype(np.float32))
    return views

def extract_all_views_one(path):
    y, sr = sf.read(str(path), dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    if len(y) == 0:
        y = np.zeros(N_FFT, dtype=np.float32)
    dur = len(y) / sr
    y_pad = np.pad(y, (0, N_FFT - len(y))) if len(y) < N_FFT else y

    # Recording-level features
    S = np.abs(librosa.stft(y_pad, n_fft=N_FFT, hop_length=HOP))
    P2 = S ** 2
    rms  = librosa.feature.rms(y=y_pad, frame_length=N_FFT, hop_length=HOP)
    zcr  = librosa.feature.zero_crossing_rate(y=y_pad, frame_length=N_FFT, hop_length=HOP)
    mel128 = librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=128)
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel128), n_mfcc=20)
    chroma = librosa.feature.chroma_stft(S=P2, sr=sr)
    cent = librosa.feature.spectral_centroid(S=S, sr=sr)
    bw   = librosa.feature.spectral_bandwidth(S=S, sr=sr)
    roll = librosa.feature.spectral_rolloff(S=S, sr=sr)
    con  = librosa.feature.spectral_contrast(S=S, sr=sr)
    ms = lambda M: np.concatenate([M.mean(axis=1), M.std(axis=1)])
    base = np.concatenate([ms(mfcc), ms(chroma), ms(cent), ms(bw), ms(roll), ms(zcr), ms(rms), [dur]])
    contrast = ms(con)
    r1 = rms[0]; T = S.shape[1]
    act = r1 >= max(np.percentile(r1, 30), 0.02 * r1.max())
    if act.sum() < 3:
        act = np.ones(T, bool)
    lm = librosa.power_to_db(librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=40))
    lma, mfa = lm[:, act], mfcc[:, act]
    dm = librosa.feature.delta(mfcc, width=9 if T >= 9 else 3, mode="nearest")
    flat = librosa.feature.spectral_flatness(S=S)[0]
    env = librosa.onset.onset_strength(S=lm, sr=sr, hop_length=HOP)
    n_on = count_onsets(env, sr, HOP)
    lr = np.log(r1 + 1e-8)
    rich = np.concatenate([
        lma.mean(1), lma.std(1), np.percentile(lma, 90, axis=1), lm.max(1),
        mfa.mean(1), mfa.std(1), dm.std(1),
        [flat.mean(), flat.std(), env.mean(), env.std()],
        np.percentile(lr, [10, 50, 90]),
        [act.mean(), np.log(dur + 1e-3), n_on / max(dur, 1e-3)]
    ])
    f_rich = np.concatenate([base, contrast, rich]).astype(np.float32)

    # Harmonic-percussive features
    raw_rms = float(np.sqrt(np.mean(y * y)) + 1e-8)
    raw_peak = float(np.max(np.abs(y)) + 1e-8)
    S_n = S / raw_peak
    P_n = S_n ** 2
    S_h, S_p = librosa.decompose.hpss(S_n, margin=2.0)
    P_h, P_p = S_h ** 2, S_p ** 2
    fe = P_n.sum(axis=0)
    fe_db = 10.0 * np.log10(fe + 1e-10)
    act20 = fe_db >= (fe_db.max() - 20.0)
    if act20.sum() < 4: act20 = fe >= np.percentile(fe, 70)
    if act20.sum() < 2: act20 = np.ones(len(fe), bool)
    top20 = fe >= np.percentile(fe, 80)
    if top20.sum() < 2: top20 = np.ones(len(fe), bool)
    h_e = P_h.sum(0); p_e = P_p.sum(0); t_e = fe + 1e-10
    h_rat = h_e / t_e; p_rat = p_e / t_e
    mel_h32 = librosa.power_to_db(MEL32_2048 @ P_h, ref=np.max)
    mel_p32 = librosa.power_to_db(MEL32_2048 @ P_p, ref=np.max)
    mfcc_h = librosa.feature.mfcc(S=mel_h32, n_mfcc=16)
    mfcc_p = librosa.feature.mfcc(S=mel_p32, n_mfcc=16)
    cent_h = librosa.feature.spectral_centroid(S=S_h, sr=sr)[0]
    cent_p = librosa.feature.spectral_centroid(S=S_p, sr=sr)[0]
    flat_h = librosa.feature.spectral_flatness(S=S_h + 1e-8)[0]
    flat_p = librosa.feature.spectral_flatness(S=S_p + 1e-8)[0]
    def stats3(M, mask):
        A = mask_slice(M, mask)
        return np.concatenate([A.mean(1), A.std(1), np.percentile(A, 90, axis=1)])
    f_hpss = np.concatenate([
        [h_rat.mean(), h_rat.std(), h_rat[act20].mean(), p_rat.mean(), p_rat.std(), p_rat[act20].mean(),
         (h_e.sum() / (t_e.sum() + 1e-10)), (p_e.sum() / (t_e.sum() + 1e-10)),
         cent_h[act20].mean(), cent_h[act20].std(), cent_p[act20].mean(), cent_p[act20].std(),
         flat_h[act20].mean(), flat_p[act20].mean(),
         np.log(raw_rms), np.log(raw_peak), raw_peak / raw_rms, act20.mean()],
        stats3(mfcc_h, act20), stats3(mfcc_p, act20),
        mask_slice(mel_h32, top20).mean(1), mask_slice(mel_p32, top20).mean(1),
    ]).astype(np.float32)

    # Modulation spectrum features
    width_m = 1024
    hop_m = max(1, round(sr * 0.010))
    yp_m = np.pad(y, (width_m // 2, width_m // 2))
    frames_m = np.lib.stride_tricks.sliding_window_view(yp_m, width_m)[::hop_m]
    spec_m = np.abs(np.fft.rfft(frames_m * np.hanning(width_m), axis=1)) ** 2
    hz = np.fft.rfftfreq(width_m, 1.0 / sr)
    mel_edges = np.linspace(0, np.log1p(min(11000, sr / 2) / 700), 26)
    edges = 700 * np.expm1(mel_edges)
    bank = np.maximum(0, np.minimum(
        (hz[:, None] - edges[:-2]) / np.diff(edges)[:-1],
        (edges[2:] - hz[:, None]) / np.diff(edges)[1:]))
    energy = spec_m @ bank
    envelope = np.log1p(100 * energy / max(energy.mean(), 1e-10))
    centered = envelope - envelope.mean(axis=0)
    scales = np.sqrt(np.mean(centered ** 2, axis=0))
    norm = centered / np.maximum(scales, 1e-6)
    corr = norm.T @ norm / len(norm)
    freq, power = periodogram(centered, fs=sr / hop_m, axis=0, nfft=max(256, len(centered)), detrend=False, scaling="density")
    bands = [(0.5, 2), (2, 4), (4, 8), (8, 16), (16, 32), (32, 50)]
    modulation = [np.log1p(power[(freq >= lo) & (freq < hi)].sum(0) * freq[1]) for lo, hi in bands]
    f_mod = np.concatenate([envelope.mean(0), scales, corr[np.triu_indices(24, k=1)], *modulation]).astype(np.float32)

    # Short-window spectrogram patches
    hop_p = max(1, round(sr * 0.012))
    S1 = np.abs(librosa.stft(y, n_fft=1024, hop_length=hop_p))
    P1 = S1 ** 2
    lm40 = librosa.power_to_db(MEL40_1024 @ P1, ref=np.max).T
    p_std = make_window_patches(lm40, (8, 32))

    S1_h, S1_p = librosa.decompose.hpss(S1, margin=2.0)
    lm_h = librosa.power_to_db(MEL24_1024 @ (S1_h ** 2), ref=np.max).T
    lm_p = librosa.power_to_db(MEL24_1024 @ (S1_p ** 2), ref=np.max).T
    p_hpss = make_window_patches(np.hstack([lm_h, lm_p]), (8, 32))

    # Harmonic-percussive modulation features
    eh = (MEL16_1024 @ (S1_h ** 2)).T
    ep = (MEL16_1024 @ (S1_p ** 2)).T
    env_h = np.log1p(100.0 * eh / max(eh.mean(), 1e-10))
    env_p = np.log1p(100.0 * ep / max(ep.mean(), 1e-10))
    ch = env_h - env_h.mean(0); cp = env_p - env_p.mean(0)
    sh = np.sqrt(np.mean(ch ** 2, axis=0)); sp = np.sqrt(np.mean(cp ** 2, axis=0))
    nh = ch / np.maximum(sh, 1e-6); np_ = cp / np.maximum(sp, 1e-6)
    corr_h = (nh.T @ nh) / len(nh); corr_p = (np_.T @ np_) / len(np_)
    corr_hp_diag = np.mean(nh * np_, axis=0)
    fs_env = sr / float(hop_p)
    fq, pw_h = periodogram(ch, fs=fs_env, axis=0, nfft=max(256, len(ch)), detrend=False, scaling="density")
    _,  pw_p = periodogram(cp, fs=fs_env, axis=0, nfft=max(256, len(cp)), detrend=False, scaling="density")
    mbands_hp = [(0.5, 2), (2, 4), (4, 8), (8, 16), (16, 32), (32, 41.5)]
    mod_h = [np.log1p(pw_h[(fq >= lo) & (fq < hi)].sum(0) * fq[1]) for lo, hi in mbands_hp]
    mod_p = [np.log1p(pw_p[(fq >= lo) & (fq < hi)].sum(0) * fq[1]) for lo, hi in mbands_hp]
    f_mod_hpss = np.concatenate([
        env_h.mean(0), sh, env_p.mean(0), sp,
        corr_h[np.triu_indices(16, k=1)], corr_p[np.triu_indices(16, k=1)], corr_hp_diag,
        *mod_h, *mod_p
    ]).astype(np.float32)

    return f_rich, f_hpss, f_mod, f_mod_hpss, p_std, p_hpss

def transform_patch_family(patch_list, encoders):

    boaw_v, vlad_v = [], []
    n_gmm_seeds = len(encoders[0]["gmms"])
    fv_by_seed = [[] for _ in range(n_gmm_seeds)]
    for view, enc in enumerate(encoders):
        scaler = enc["scaler"]
        c96 = enc["c96"]
        c32 = enc["c32"]
        def _boaw(rec):
            fr = scaler.transform(rec[view]).astype(np.float32)
            d = cdist(fr, c96, "sqeuclidean")
            nr = np.argpartition(d, 2, axis=1)[:, :3]
            w = 1.0 / np.maximum(np.take_along_axis(d, nr, axis=1), 1e-8)
            w /= w.sum(axis=1, keepdims=True)
            h = np.bincount(nr.ravel(), weights=w.ravel(), minlength=len(c96))
            return np.sqrt(h / max(h.sum(), 1e-12)).astype(np.float32)
        boaw_v.append(np.vstack([_boaw(r) for r in patch_list]))

        def _vlad(rec):
            fr = scaler.transform(rec[view]).astype(np.float32)
            asgn = cdist(fr, c32, "sqeuclidean").argmin(axis=1)
            sums = np.zeros_like(c32)
            np.add.at(sums, asgn, fr - c32[asgn])
            sums /= np.maximum(np.linalg.norm(sums, axis=1, keepdims=True), 1e-8)
            v = np.sign(sums.ravel()) * np.sqrt(np.abs(sums.ravel()))
            return (v / max(np.linalg.norm(v), 1e-8)).astype(np.float32)
        vlad_v.append(np.stack([_vlad(r) for r in patch_list]))

        for si, g_enc in enumerate(enc["gmms"]):
            pca = g_enc["pca"]; gmm = g_enc["gmm"]
            mu, var, wt = gmm.means_, gmm.covariances_, gmm.weights_
            def _fv(rec):
                x = pca.transform(scaler.transform(rec[view]))
                q = gmm.predict_proba(x)
                mass = q.sum(axis=0)[:, None]
                mom = q.T @ x
                norm = len(x) * np.sqrt(wt[:, None])
                f1 = (mom - mass * mu) / (norm * np.sqrt(var))
                f2 = ((q.T @ (x * x) - 2 * mu * mom + mass * mu**2) / var - mass) / (np.sqrt(2) * norm)
                v = np.concatenate([f1.ravel(), f2.ravel()])
                v = np.sign(v) * np.sqrt(np.abs(v))
                return (v / max(np.linalg.norm(v), 1e-8)).astype(np.float32)
            fv_by_seed[si].append(np.stack([_fv(r) for r in patch_list]))
    return (
        np.hstack(boaw_v).astype(np.float32),
        np.hstack(vlad_v).astype(np.float32),
        [np.hstack(fv_by_seed[si]).astype(np.float32) for si in range(n_gmm_seeds)],
    )


def fit_patch_family(patch_list, pca_dim, n_gmm, gmm_seeds, max_per_clip=24, base_seed=42):

    encoders = []
    for view in range(2):
        rng = np.random.default_rng(base_seed + view)
        pool = np.vstack([
            patch_list[i][view][rng.choice(len(patch_list[i][view]), min(max_per_clip, len(patch_list[i][view])), replace=False)]
            for i in unc_ids
        ])
        scaler = StandardScaler().fit(pool)
        pool_s = scaler.transform(pool).astype(np.float32)

        km96 = MiniBatchKMeans(n_clusters=96, batch_size=2048, n_init=3, random_state=42).fit(pool_s)
        c96 = km96.cluster_centers_.astype(np.float32)

        km32 = MiniBatchKMeans(n_clusters=32, batch_size=2048, n_init=3, random_state=17).fit(pool_s)
        c32 = km32.cluster_centers_.astype(np.float32)

        gmms = []
        for gs in gmm_seeds:
            pca = PCA(n_components=pca_dim, whiten=True, svd_solver="randomized", random_state=gs)
            proj = pca.fit_transform(pool_s)
            gmm = GaussianMixture(n_gmm, covariance_type="diag", reg_covar=0.01, max_iter=80, random_state=gs).fit(proj)
            gmms.append({"pca": pca, "gmm": gmm})
        encoders.append({"scaler": scaler, "c96": c96, "c32": c32, "gmms": gmms})

    boaw, vlad, fvs = transform_patch_family(patch_list, encoders)
    return encoders, boaw, vlad, fvs

t_ext = time.time()
train_paths = [TRAIN_AUDIO / f"{i}.wav" for i in train_df["id"]]
print(f"Extracting features for {len(train_paths)} training clips...", flush=True)
res = Parallel(n_jobs=N_JOBS, batch_size=8, verbose=5)(delayed(extract_all_views_one)(p) for p in train_paths)
print(f"Feature extraction: {time.time() - t_ext:.1f}s", flush=True)

X_rich_tr = np.vstack([r[0] for r in res]).astype(np.float32)
X_hpss_tr = np.vstack([r[1] for r in res]).astype(np.float32)
X_mod_tr  = np.vstack([r[2] for r in res]).astype(np.float32)
X_mh_tr   = np.vstack([r[3] for r in res]).astype(np.float32)
std_patches_tr  = [r[4] for r in res]
hpss_patches_tr = [r[5] for r in res]
del res; gc.collect()

X_rich_tr = np.where(np.isfinite(X_rich_tr), X_rich_tr, np.nan)
med_r = np.nanmedian(X_rich_tr, axis=0).astype(np.float32)
X_rich_tr = np.where(np.isnan(X_rich_tr), med_r, X_rich_tr).astype(np.float32)
X_hpss_tr = np.nan_to_num(X_hpss_tr).astype(np.float32)
X_mod_tr  = np.nan_to_num(X_mod_tr).astype(np.float32)
X_mh_tr   = np.nan_to_num(X_mh_tr).astype(np.float32)
X_tab_tr  = np.hstack([X_rich_tr, X_hpss_tr]).astype(np.float32)

print("Encoding log-mel patches...", flush=True)
std_encoders, boaw_s_tr, vlad_s_tr, [fv_s_tr] = fit_patch_family(std_patches_tr, pca_dim=24, n_gmm=32, gmm_seeds=[81], max_per_clip=20, base_seed=42)
del std_patches_tr; gc.collect()

print("Encoding HPSS patches...", flush=True)
hpss_encoders, boaw_h_tr, vlad_h_tr, [fv_h1_tr, fv_h2_tr] = fit_patch_family(hpss_patches_tr, pca_dim=32, n_gmm=40, gmm_seeds=[81, 137], max_per_clip=24, base_seed=42)
del hpss_patches_tr; gc.collect()

print(f"Views ready ({time.time() - T_GLOBAL:.0f}s): rich={X_rich_tr.shape[1]}, hpss={X_hpss_tr.shape[1]}, mod={X_mod_tr.shape[1]}, mod_hpss={X_mh_tr.shape[1]}")

## Kernel matrices

In [ ]:
K_fv_s = linear_kernel(fv_s_tr, fv_s_tr).astype(np.float32)
K_vl_s = linear_kernel(vlad_s_tr, vlad_s_tr).astype(np.float32)
K_fv_h = (0.5 * linear_kernel(fv_h1_tr, fv_h1_tr) + 0.5 * linear_kernel(fv_h2_tr, fv_h2_tr)).astype(np.float32)
K_vl_h = linear_kernel(vlad_h_tr, vlad_h_tr).astype(np.float32)

boaw_both_tr = np.hstack([boaw_s_tr, boaw_h_tr]).astype(np.float32)
boaw_both_gamma = float(1.0 / (boaw_both_tr.shape[1] * boaw_both_tr.var()))
boaw_s_gamma    = float(1.0 / (boaw_s_tr.shape[1] * boaw_s_tr.var()))
K_bw_all = rbf_kernel(boaw_both_tr, boaw_both_tr, gamma=boaw_both_gamma).astype(np.float32)
K_bw_s   = rbf_kernel(boaw_s_tr, boaw_s_tr, gamma=boaw_s_gamma).astype(np.float32)

X_jlr_tr = np.hstack([
    np.sqrt(0.28) * fv_h1_tr,
    np.sqrt(0.28) * fv_h2_tr,
    np.sqrt(0.26) * fv_s_tr,
    np.sqrt(0.18) * vlad_h_tr,
    np.sqrt(0.14) * vlad_s_tr,
]).astype(np.float32)
print(f"Gram matrices and linear features {X_jlr_tr.shape} ready.")

## Cross-validation and calibration

In [ ]:
def map3_s(yi, P):
    top3 = np.argsort(-P, axis=1)[:, :3]
    return ((top3 == yi[:, None]) / np.array([1.0, 2.0, 3.0])).sum(1)

def sharpen(P, t):
    Q = np.clip(P, 1e-9, None) ** t
    return Q / Q.sum(1, keepdims=True)

def fit_temp(P, yi, w):
    best = (1e9, 1.0)
    for t in np.linspace(0.5, 4.0, 36):
        Q = sharpen(P, t)
        nll = -np.average(np.log(Q[np.arange(len(yi)), yi] + 1e-12), weights=w)
        if nll < best[0]:
            best = (nll, t)
    return float(best[1])

def make_tab_specs(n_jobs=1):
    return [
        (make_pipeline(StandardScaler(), SVC(C=3, kernel="rbf", class_weight="balanced", probability=True, random_state=42)), "svc__sample_weight", 0.25),
        (make_pipeline(StandardScaler(), SVC(C=10, kernel="rbf", class_weight="balanced", probability=True, random_state=42)), "svc__sample_weight", 0.15),
        (make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)), "logisticregression__sample_weight", 0.10),
        (RandomForestClassifier(n_estimators=500, class_weight="balanced", min_samples_leaf=2, random_state=42, n_jobs=n_jobs), "sample_weight", 0.25),
        (ExtraTreesClassifier(n_estimators=500, class_weight="balanced", min_samples_leaf=2, random_state=42, n_jobs=n_jobs), "sample_weight", 0.25),
    ]

def fit_predict_tab(Xtr, ytr, wtr, X_list, n_jobs=1):
    outs = [np.zeros((len(X), K), dtype=np.float64) for X in X_list]
    for m, wk, a in make_tab_specs(n_jobs):
        if wtr is None: m.fit(Xtr, ytr)
        else:           m.fit(Xtr, ytr, **{wk: wtr})
        for o, X in zip(outs, X_list):
            o[:, m.classes_] += a * m.predict_proba(X)
    return outs

def fit_stage1_teacher(v_tr_ids, y_tr, target_ids, n_jobs_tab=1):
    p_tab = fit_predict_tab(X_tab_tr[v_tr_ids], y_tr, None, [X_tab_tr[target_ids]], n_jobs=n_jobs_tab)[0]
    sc_m = StandardScaler().fit(X_mod_tr[v_tr_ids])
    p_mod = SVC(C=8, kernel="rbf", class_weight="balanced", probability=True, random_state=42).fit(
        sc_m.transform(X_mod_tr[v_tr_ids]), y_tr).predict_proba(sc_m.transform(X_mod_tr[target_ids]))
    K_t = 0.42 * K_fv_h + 0.24 * K_fv_s + 0.18 * K_vl_h + 0.10 * K_vl_s + 0.06 * K_bw_all
    p_k = SVC(kernel="precomputed", C=12, class_weight="balanced", probability=True, random_state=42).fit(
        K_t[np.ix_(v_tr_ids, v_tr_ids)], y_tr).predict_proba(K_t[np.ix_(target_ids, v_tr_ids)])
    p_lr = LogisticRegression(C=10.0, class_weight="balanced", max_iter=400).fit(
        X_jlr_tr[v_tr_ids], y_tr).predict_proba(X_jlr_tr[target_ids])
    return 0.28 * p_tab + 0.14 * p_mod + 0.34 * p_k + 0.24 * p_lr

def fit_eval_fold_streams(v_tr_ids, y_tr, target_ids, keep, n_jobs_tab=1):
    all_ids  = np.r_[v_tr_ids, unc_ids]
    all_y    = np.r_[y_tr, yu]
    all_w    = np.r_[np.ones(len(v_tr_ids)), np.where(keep, 0.50, 0.08)]
    keep_ids = np.r_[v_tr_ids, unc_ids[keep]]
    keep_y   = np.r_[y_tr, yu[keep]]
    keep_w   = np.r_[np.ones(len(v_tr_ids)), np.full(keep.sum(), 0.50)]

    sc_r = StandardScaler().fit(X_rich_tr[all_ids])
    sc_h = StandardScaler().fit(X_hpss_tr[all_ids])
    sc_m = StandardScaler().fit(X_mod_tr[all_ids])
    Xr_tr, Xr_tg = sc_r.transform(X_rich_tr[all_ids]), sc_r.transform(X_rich_tr[target_ids])
    Xh_tr, Xh_tg = sc_h.transform(X_hpss_tr[all_ids]), sc_h.transform(X_hpss_tr[target_ids])
    Xm_tr, Xm_tg = sc_m.transform(X_mod_tr[all_ids]),  sc_m.transform(X_mod_tr[target_ids])

    Kr_tr = rbf_kernel(Xr_tr, Xr_tr, gamma=1.0 / Xr_tr.shape[1]).astype(np.float32)
    Kr_tg = rbf_kernel(Xr_tg, Xr_tr, gamma=1.0 / Xr_tr.shape[1]).astype(np.float32)
    Kh_tr = rbf_kernel(Xh_tr, Xh_tr, gamma=1.0 / Xh_tr.shape[1]).astype(np.float32)
    Kh_tg = rbf_kernel(Xh_tg, Xh_tr, gamma=1.0 / Xh_tr.shape[1]).astype(np.float32)
    Km_tr = rbf_kernel(Xm_tr, Xm_tr, gamma=1.0 / Xm_tr.shape[1]).astype(np.float32)
    Km_tg = rbf_kernel(Xm_tg, Xm_tr, gamma=1.0 / Xm_tr.shape[1]).astype(np.float32)

    K_mkl_tr = (0.24 * K_fv_h[np.ix_(all_ids, all_ids)] + 0.16 * K_fv_s[np.ix_(all_ids, all_ids)] +
                0.14 * K_vl_h[np.ix_(all_ids, all_ids)] + 0.10 * K_vl_s[np.ix_(all_ids, all_ids)] +
                0.06 * K_bw_all[np.ix_(all_ids, all_ids)] + 0.11 * Kr_tr + 0.10 * Kh_tr + 0.09 * Km_tr)
    K_mkl_tg = (0.24 * K_fv_h[np.ix_(target_ids, all_ids)] + 0.16 * K_fv_s[np.ix_(target_ids, all_ids)] +
                0.14 * K_vl_h[np.ix_(target_ids, all_ids)] + 0.10 * K_vl_s[np.ix_(target_ids, all_ids)] +
                0.06 * K_bw_all[np.ix_(target_ids, all_ids)] + 0.11 * Kr_tg + 0.10 * Kh_tg + 0.09 * Km_tg)
    p_mkl = SVC(kernel="precomputed", C=12, probability=True, random_state=42).fit(K_mkl_tr, all_y, sample_weight=all_w).predict_proba(K_mkl_tg)
    p_jlr = LogisticRegression(C=12.0, max_iter=500).fit(X_jlr_tr[all_ids], all_y, sample_weight=all_w).predict_proba(X_jlr_tr[target_ids])

    K_hp_tr = 0.60 * K_fv_h[np.ix_(all_ids, all_ids)] + 0.40 * K_vl_h[np.ix_(all_ids, all_ids)]
    K_hp_tg = 0.60 * K_fv_h[np.ix_(target_ids, all_ids)] + 0.40 * K_vl_h[np.ix_(target_ids, all_ids)]
    p_hpss_k = SVC(kernel="precomputed", C=12, probability=True, random_state=42).fit(K_hp_tr, all_y, sample_weight=all_w).predict_proba(K_hp_tg)

    K_st_tr = 0.55 * K_fv_s[np.ix_(all_ids, all_ids)] + 0.30 * K_vl_s[np.ix_(all_ids, all_ids)] + 0.15 * K_bw_s[np.ix_(all_ids, all_ids)]
    K_st_tg = 0.55 * K_fv_s[np.ix_(target_ids, all_ids)] + 0.30 * K_vl_s[np.ix_(target_ids, all_ids)] + 0.15 * K_bw_s[np.ix_(target_ids, all_ids)]
    p_std_k = SVC(kernel="precomputed", C=12, probability=True, random_state=42).fit(K_st_tr, all_y, sample_weight=all_w).predict_proba(K_st_tg)

    p_tab = fit_predict_tab(X_tab_tr[keep_ids], keep_y, keep_w, [X_tab_tr[target_ids]], n_jobs=n_jobs_tab)[0]

    sc_mk = StandardScaler().fit(X_mod_tr[keep_ids])
    p_mod_k = SVC(C=8, kernel="rbf", probability=True, random_state=42).fit(
        sc_mk.transform(X_mod_tr[keep_ids]), keep_y, sample_weight=keep_w).predict_proba(sc_mk.transform(X_mod_tr[target_ids]))
    p_mod_a = SVC(kernel="precomputed", C=10, probability=True, random_state=42).fit(Km_tr, all_y, sample_weight=all_w).predict_proba(Km_tg)
    p_mod = 0.50 * p_mod_k + 0.50 * p_mod_a

    sc_mhk = StandardScaler().fit(X_mh_tr[keep_ids])
    p_mhk = SVC(C=8, kernel="rbf", probability=True, random_state=42).fit(
        sc_mhk.transform(X_mh_tr[keep_ids]), keep_y, sample_weight=keep_w).predict_proba(sc_mhk.transform(X_mh_tr[target_ids]))
    sc_mha = StandardScaler().fit(X_mh_tr[all_ids])
    Xmha_tr, Xmha_tg = sc_mha.transform(X_mh_tr[all_ids]), sc_mha.transform(X_mh_tr[target_ids])
    Kmha_tr = rbf_kernel(Xmha_tr, Xmha_tr, gamma=1.0 / Xmha_tr.shape[1]).astype(np.float32)
    Kmha_tg = rbf_kernel(Xmha_tg, Xmha_tr, gamma=1.0 / Xmha_tr.shape[1]).astype(np.float32)
    p_mha = SVC(kernel="precomputed", C=10, probability=True, random_state=42).fit(Kmha_tr, all_y, sample_weight=all_w).predict_proba(Kmha_tg)
    p_mod_hpss = 0.50 * p_mhk + 0.50 * p_mha

    lda = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))
    lda.fit(X_tab_tr[keep_ids], keep_y)
    p_lda = np.zeros((len(target_ids), K), dtype=np.float64)
    p_lda[:, lda.classes_.astype(int)] = lda.predict_proba(X_tab_tr[target_ids])

    n_vtr = len(v_tr_ids)
    S_v = K_mkl_tg[:, :n_vtr]
    cnt_tr = np.bincount(y_tr, minlength=K)
    inv_c = (1.0 / np.maximum(cnt_tr, 1))[y_tr]
    p_knn = np.zeros((len(target_ids), K), dtype=np.float64)
    for i in range(len(target_ids)):
        sims = S_v[i]
        top = np.argpartition(-sims, 16)[:16]
        w_n = np.exp(15.0 * (sims[top] - sims[top].max())) * inv_c[top]
        votes = np.bincount(y_tr[top], weights=w_n, minlength=K)
        p_knn[i] = votes / np.maximum(votes.sum(), 1e-12)

    return dict(mkl=p_mkl, jlr=p_jlr, hpss_k=p_hpss_k, std_k=p_std_k, tab=p_tab, mod=p_mod, mod_hpss=p_mod_hpss, lda=p_lda, knn=p_knn)

W_FIXED = dict(mkl=0.25, jlr=0.16, hpss_k=0.14, std_k=0.09, tab=0.11, mod=0.05, mod_hpss=0.09, lda=0.04, knn=0.07)
oofs_s1 = {}
blend_raw_by_seed = {}
splits_by_seed = {}

for seed in SEEDS:
    ts = time.time()
    splits = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(ver_ids, yv))
    splits_by_seed[seed] = splits
    s1_res = Parallel(n_jobs=N_JOBS)(
        delayed(fit_stage1_teacher)(ver_ids[tr_i], yv[tr_i], np.r_[ver_ids[va_i], unc_ids])
        for tr_i, va_i in splits
    )
    oof_t = np.zeros((len(yv), K))
    Pu_list = []
    for (tr_i, va_i), P_out in zip(splits, s1_res):
        oof_t[va_i] = P_out[:len(va_i)]
        Pu_list.append(P_out[len(va_i):])
    oofs_s1[seed] = oof_t

    def run_fold(fi):
        tr_i, va_i = splits[fi]
        Pu = Pu_list[fi]
        wu = w_unif[tr_i] / w_unif[tr_i].sum()
        pi_t = (oof_t[tr_i] * wu[:, None]).sum(0)
        Pu_deb = Pu / np.maximum(pi_t[None, :], 1e-6)
        Pu_deb /= Pu_deb.sum(1, keepdims=True)
        keep = (Pu_deb > Pu_deb[np.arange(len(yu)), yu][:, None]).sum(1) < 1
        d = fit_eval_fold_streams(ver_ids[tr_i], yv[tr_i], ver_ids[va_i], keep, n_jobs_tab=1)
        return sum(W_FIXED[nm] * d[nm] for nm in W_FIXED)

    fold_preds = Parallel(n_jobs=N_JOBS)(delayed(run_fold)(fi) for fi in range(5))
    oof_b = np.zeros((len(yv), K))
    for (_, va_i), P_va in zip(splits, fold_preds):
        oof_b[va_i] = P_va
    blend_raw_by_seed[seed] = oof_b

    oof_deb = np.zeros_like(oof_b)
    for tr_i, va_i in splits:
        wu = w_unif[tr_i] / w_unif[tr_i].sum()
        pi_m = (oof_b[tr_i] * wu[:, None]).sum(0)
        Q = oof_b[va_i] / np.maximum(pi_m[None, :], 1e-6)
        oof_deb[va_i] = Q / Q.sum(1, keepdims=True)
    sc = map3_s(yv, oof_deb)
    print(f"Seed {seed} ({time.time()-ts:.0f}s) | plain={sc.mean():.5f} | unif={np.average(sc,weights=w_unif):.5f} | w_tot={np.average(sc,weights=w_tot):.5f}", flush=True)

wu_all = w_unif / w_unif.sum()
P_oof_mean = np.mean([blend_raw_by_seed[s] for s in SEEDS], axis=0)
pi_src = (P_oof_mean * wu_all[:, None]).sum(0)
P_oof_deb = P_oof_mean / np.maximum(pi_src[None, :], 1e-6)
P_oof_deb /= P_oof_deb.sum(1, keepdims=True)
temp_opt = fit_temp(P_oof_deb, yv, w_unif)
print(f"Calibration temperature: {temp_opt:.2f}")

## Final training

In [ ]:
t_fin = time.time()

# Stage-1 teacher on all verified clips
oof_s1_mean = np.mean([oofs_s1[s] for s in SEEDS], axis=0)
pi_s1 = (oof_s1_mean * wu_all[:, None]).sum(0)
Pu_full = fit_stage1_teacher(ver_ids, yv, unc_ids, n_jobs_tab=N_JOBS)
Pu_full_deb = Pu_full / np.maximum(pi_s1[None, :], 1e-6)
Pu_full_deb /= Pu_full_deb.sum(1, keepdims=True)
keep_full = (Pu_full_deb > Pu_full_deb[np.arange(len(yu)), yu][:, None]).sum(1) < 1
print(f"Retained unchecked clips: {keep_full.sum()}/{len(yu)} ({100*keep_full.mean():.1f}%)", flush=True)

all_ids  = np.r_[ver_ids, unc_ids]
all_y    = np.r_[yv, yu]
all_w    = np.r_[np.ones(len(ver_ids)), np.where(keep_full, 0.50, 0.08)]
keep_ids = np.r_[ver_ids, unc_ids[keep_full]]
keep_y   = np.r_[yv, yu[keep_full]]
keep_w   = np.r_[np.ones(len(ver_ids)), np.full(keep_full.sum(), 0.50)]

# Fit scalers
sc_r   = StandardScaler().fit(X_rich_tr[all_ids])
sc_h   = StandardScaler().fit(X_hpss_tr[all_ids])
sc_m   = StandardScaler().fit(X_mod_tr[all_ids])
sc_mk  = StandardScaler().fit(X_mod_tr[keep_ids])
sc_mhk = StandardScaler().fit(X_mh_tr[keep_ids])
sc_mha = StandardScaler().fit(X_mh_tr[all_ids])

Xr_all_s   = sc_r.transform(X_rich_tr[all_ids]).astype(np.float32)
Xh_all_s   = sc_h.transform(X_hpss_tr[all_ids]).astype(np.float32)
Xm_all_s   = sc_m.transform(X_mod_tr[all_ids]).astype(np.float32)
Xmha_all_s = sc_mha.transform(X_mh_tr[all_ids]).astype(np.float32)

Kr_tr   = rbf_kernel(Xr_all_s, Xr_all_s, gamma=1.0 / Xr_all_s.shape[1]).astype(np.float32)
Kh_tr   = rbf_kernel(Xh_all_s, Xh_all_s, gamma=1.0 / Xh_all_s.shape[1]).astype(np.float32)
Km_tr   = rbf_kernel(Xm_all_s, Xm_all_s, gamma=1.0 / Xm_all_s.shape[1]).astype(np.float32)
Kmha_tr = rbf_kernel(Xmha_all_s, Xmha_all_s, gamma=1.0 / Xmha_all_s.shape[1]).astype(np.float32)

# Fit final models
print("Fitting final models...", flush=True)
K_mkl_tr = (0.24 * K_fv_h[np.ix_(all_ids, all_ids)] + 0.16 * K_fv_s[np.ix_(all_ids, all_ids)] +
            0.14 * K_vl_h[np.ix_(all_ids, all_ids)] + 0.10 * K_vl_s[np.ix_(all_ids, all_ids)] +
            0.06 * K_bw_all[np.ix_(all_ids, all_ids)] + 0.11 * Kr_tr + 0.10 * Kh_tr + 0.09 * Km_tr)
clf_mkl = SVC(kernel="precomputed", C=12, probability=True, random_state=42).fit(K_mkl_tr, all_y, sample_weight=all_w)

clf_jlr = LogisticRegression(C=12.0, max_iter=500).fit(X_jlr_tr[all_ids], all_y, sample_weight=all_w)

K_hp_tr = 0.60 * K_fv_h[np.ix_(all_ids, all_ids)] + 0.40 * K_vl_h[np.ix_(all_ids, all_ids)]
clf_hpss_k = SVC(kernel="precomputed", C=12, probability=True, random_state=42).fit(K_hp_tr, all_y, sample_weight=all_w)

K_st_tr = 0.55 * K_fv_s[np.ix_(all_ids, all_ids)] + 0.30 * K_vl_s[np.ix_(all_ids, all_ids)] + 0.15 * K_bw_s[np.ix_(all_ids, all_ids)]
clf_std_k = SVC(kernel="precomputed", C=12, probability=True, random_state=42).fit(K_st_tr, all_y, sample_weight=all_w)

tab_models = []
for m, wk, a in make_tab_specs(N_JOBS):
    m.fit(X_tab_tr[keep_ids], keep_y, **{wk: keep_w})
    tab_models.append((m, a))

clf_mod_k = SVC(C=8, kernel="rbf", probability=True, random_state=42).fit(sc_mk.transform(X_mod_tr[keep_ids]), keep_y, sample_weight=keep_w)
clf_mod_a = SVC(kernel="precomputed", C=10, probability=True, random_state=42).fit(Km_tr, all_y, sample_weight=all_w)

clf_mhk = SVC(C=8, kernel="rbf", probability=True, random_state=42).fit(sc_mhk.transform(X_mh_tr[keep_ids]), keep_y, sample_weight=keep_w)
clf_mha = SVC(kernel="precomputed", C=10, probability=True, random_state=42).fit(Kmha_tr, all_y, sample_weight=all_w)

clf_lda = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))
clf_lda.fit(X_tab_tr[keep_ids], keep_y)
print(f"Model training: {time.time() - t_fin:.1f}s", flush=True)

# Save artifacts
meta_and_encoders = {
    "feature_version": "step2b-v1",
    "classes": classes,
    "cls_idx": cls_idx,
    "K": K,
    "med_r": med_r,
    "n_vtr": len(ver_ids),
    "yv": yv,
    "boaw_both_gamma": boaw_both_gamma,
    "boaw_s_gamma": boaw_s_gamma,
    "std_encoders": std_encoders,
    "hpss_encoders": hpss_encoders,
    "W_FIXED": W_FIXED,
    "pi_src": pi_src,
    "temp_opt": temp_opt,
}
models_bundle = {
    "sc_r": sc_r, "sc_h": sc_h, "sc_m": sc_m, "sc_mk": sc_mk, "sc_mhk": sc_mhk, "sc_mha": sc_mha,
    "clf_mkl": clf_mkl, "clf_jlr": clf_jlr, "clf_hpss_k": clf_hpss_k, "clf_std_k": clf_std_k,
    "tab_models": tab_models, "clf_mod_k": clf_mod_k, "clf_mod_a": clf_mod_a,
    "clf_mhk": clf_mhk, "clf_mha": clf_mha, "clf_lda": clf_lda,
}

joblib.dump(meta_and_encoders, ART_DIR / "meta_and_encoders.joblib", compress=3)
joblib.dump(models_bundle, ART_DIR / "models.joblib", compress=3)
np.savez(
    ART_DIR / "train_refs.npz",
    Xr_all_s=Xr_all_s, Xh_all_s=Xh_all_s, Xm_all_s=Xm_all_s, Xmha_all_s=Xmha_all_s,
    fv_h1_all=fv_h1_tr[all_ids], fv_h2_all=fv_h2_tr[all_ids], fv_s_all=fv_s_tr[all_ids],
    vl_h_all=vlad_h_tr[all_ids], vl_s_all=vlad_s_tr[all_ids],
    bw_both_all=boaw_both_tr[all_ids], bw_s_all=boaw_s_tr[all_ids],
)

total_mb = sum(p.stat().st_size for p in ART_DIR.glob("*")) / (1024 ** 2)
print("Saved artifacts:")
for p in sorted(ART_DIR.glob("*")):
    print(f"  {p.name:30s} : {p.stat().st_size / (1024**2):7.1f} MB")
print(f"Total size: {total_mb:.1f} MB | elapsed: {(time.time() - T_GLOBAL)/60:.1f} min")

## Summary and evaluation

### Implementation flow (Submission 14 — Decoupled Training)
1. **Train-only extraction & encoder fitting:** Extracts all 6 acoustic views strictly on the 5,604 training clips and fits all BoAW/VLAD/PCA+GMM encoders, training medians (`med_r`), and BoAW RBF gammas on training audio only.
2. **3-seed CV calibration & full-data training:** Runs 3-seed 5-fold CV to lock `pi_src` and `temp_opt`, then retrains all 9 Step 2B streams on 100% of training data.
3. **Artifact export:** Saves `meta_and_encoders.joblib` (`0.3 MB`), `models.joblib` (`107.6 MB`), and `train_refs.npz` (`689.7 MB`) — `797.6 MB` total — to `/kaggle/working/step2b_artifacts/`.

### Changes from Submission 13 (Step 1) and all-in-one Submission 14
- Incorporates Submission 14's `mod_hpss` (`9%`) and Shrunk LDA (`4%`) upgrades over Submission 13 while pruning CatBoost/LightGBM.
- Separates offline training from test-time inference so private-test evaluation requires zero retraining and no test-set dependence during scaling/gamma estimation.

### Validation and verification
- Matches Submission 14's exact 9-stream architecture and 3-seed CV (`plain = 0.87509 | unif = 0.89170 | w_tot = 0.90390 | v10 = 0.90180`, `0.883` LB), verified locally at `99.81%` top-1 parity.